## 1.Discover the data 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mlflow
import mlflow.sklearn

In [2]:
df = pd.read_csv("data\\get_around_pricing_project.csv")

In [3]:
df.head()

,Unnamed: 0,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [4]:
# basic stats
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4843 entries, 0 to 4842
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Unnamed: 0                 4843 non-null   int64 
 1   model_key                  4843 non-null   object
 2   mileage                    4843 non-null   int64 
 3   engine_power               4843 non-null   int64 
 4   fuel                       4843 non-null   object
 5   paint_color                4843 non-null   object
 6   car_type                   4843 non-null   object
 7   private_parking_available  4843 non-null   bool  
 8   has_gps                    4843 non-null   bool  
 9   has_air_conditioning       4843 non-null   bool  
 10  automatic_car              4843 non-null   bool  
 11  has_getaround_connect      4843 non-null   bool  
 12  has_speed_regulator        4843 non-null   bool  
 13  winter_tires               4843 non-null   bool  
 14  rental_p

In [5]:
df.describe()

,Unnamed: 0,mileage,engine_power,rental_price_per_day
count,4843.000000,4.843000e+03,4843.00000,4843.000000
mean,2421.000000,1.409628e+05,128.98823,121.214536
std,1398.198007,6.019674e+04,38.99336,33.568268
min,0.000000,-6.400000e+01,0.00000,10.000000
25%,1210.500000,1.029135e+05,100.00000,104.000000
50%,2421.000000,1.410800e+05,120.00000,119.000000
75%,3631.500000,1.751955e+05,135.00000,136.000000
max,4842.000000,1.000376e+06,423.00000,422.000000


In [6]:
# check for missing values
df.isna().sum()

Unnamed: 0                   0
model_key                    0
mileage                      0
engine_power                 0
fuel                         0
paint_color                  0
car_type                     0
private_parking_available    0
has_gps                      0
has_air_conditioning         0
automatic_car                0
has_getaround_connect        0
has_speed_regulator          0
winter_tires                 0
rental_price_per_day         0
dtype: int64

## 2. Preprocessing

In [7]:
# drop column Unnamed 
df.drop(columns=df.columns[0], axis=1, inplace=True)

In [8]:
df.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [9]:
# find boolean columns
bool_cols = df.select_dtypes(include=['bool']).columns
print(bool_cols)

Index(['private_parking_available', 'has_gps', 'has_air_conditioning',
       'automatic_car', 'has_getaround_connect', 'has_speed_regulator',
       'winter_tires'],
      dtype='object')


In [10]:
# convert boolean columns to 0 and 1
df[bool_cols] = df[bool_cols].astype(int)

In [11]:
df.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,1,1,0,0,1,1,1,106
1,Citroën,13929,317,petrol,grey,convertible,1,1,0,0,0,1,1,264
2,Citroën,183297,120,diesel,white,convertible,0,0,0,0,1,0,1,101
3,Citroën,128035,135,diesel,red,convertible,1,1,0,0,1,1,1,158
4,Citroën,97097,160,diesel,silver,convertible,1,1,0,0,0,1,1,183


In [12]:
numeric_cols = df.select_dtypes(include=np.number).columns
print(numeric_cols)

Index(['mileage', 'engine_power', 'private_parking_available', 'has_gps',
       'has_air_conditioning', 'automatic_car', 'has_getaround_connect',
       'has_speed_regulator', 'winter_tires', 'rental_price_per_day'],
      dtype='object')


In [13]:
# Remove target from numeric_cols
numeric_cols = numeric_cols[: -1]
print(numeric_cols)


Index(['mileage', 'engine_power', 'private_parking_available', 'has_gps',
       'has_air_conditioning', 'automatic_car', 'has_getaround_connect',
       'has_speed_regulator', 'winter_tires'],
      dtype='object')


In [14]:
categorical_cols = df.select_dtypes(include=['object']).columns
print(categorical_cols)

Index(['model_key', 'fuel', 'paint_color', 'car_type'], dtype='object')


In [15]:
# Separate the features from the target
X = df.drop(columns=['rental_price_per_day'])
y = df['rental_price_per_day']

In [16]:
# Divide the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("...Done.")

...Done.


In [19]:
# Create preprocessor function to preprocess data we will use in the model and later for the API
def create_preprocessor(numeric_cols, categorical_cols):
    """Creates a preprocessor for the data"""
    # Preprocessor for numeric variables
    numeric_transformer = StandardScaler()
    
    # Preprocessor for categorical variables
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')
    
    # Combining preprocessors
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ])
    
    return preprocessor

In [20]:
preprocessor = create_preprocessor(numeric_cols, categorical_cols)
print("... Preprocessing done.")

... Preprocessing done.


In [17]:
# Preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [18]:
# Save the preprocessor to a file for later use in the API
joblib.dump(preprocessor, "preprocessor.joblib")

['preprocessor.joblib']

In [19]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print("...Done.")


Performing preprocessings on train set...
     model_key  mileage  engine_power    fuel paint_color car_type  \
1215   Renault   119515           135  diesel        grey   estate   
432    Citroën   234365           135  diesel       black   estate   
4244       BMW    77356           105  diesel       black      suv   
289    Peugeot   181297           105  diesel       brown   estate   
2585   Citroën   144089           137  petrol       black    sedan   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
1215                          0        1                     0              0   
432                           1        1                     0              0   
4244                          0        1                     0              0   
289                           0        1                     0              0   
2585                          1        1                     0              0   

      has_getaround_connect  has_speed_regulator  

In [20]:
print(X_train[0:5])

  (0, 0)	-0.36161786696258125
  (0, 1)	0.15047345158890083
  (0, 2)	-1.116736212353409
  (0, 3)	0.511349533896038
  (0, 4)	-0.49523606415668225
  (0, 5)	-0.49927390038334546
  (0, 6)	1.0891419780299525
  (0, 7)	-0.56522764336888
  (0, 8)	0.2693250659345363
  (0, 30)	1.0
  (0, 37)	1.0
  (0, 46)	1.0
  (0, 53)	1.0
  (1, 0)	1.5215638029960556
  (1, 1)	0.15047345158890083
  (1, 2)	0.8954666186498963
  (1, 3)	0.511349533896038
  (1, 4)	-0.49523606415668225
  (1, 5)	-0.49927390038334546
  (1, 6)	1.0891419780299525
  (1, 7)	-0.56522764336888
  (1, 8)	0.2693250659345363
  (1, 12)	1.0
  (1, 37)	1.0
  (1, 42)	1.0
  :	:
  (3, 1)	-0.6111337491628637
  (3, 2)	-1.116736212353409
  (3, 3)	0.511349533896038
  (3, 4)	-0.49523606415668225
  (3, 5)	-0.49927390038334546
  (3, 6)	1.0891419780299525
  (3, 7)	1.7691986790309506
  (3, 8)	0.2693250659345363
  (3, 28)	1.0
  (3, 37)	1.0
  (3, 44)	1.0
  (3, 53)	1.0
  (4, 0)	0.041319062576500504
  (4, 1)	0.2012472649723518
  (4, 2)	0.8954666186498963
  (4, 3)	0.511

In [21]:
# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(
    X_test
)  # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print("...Done.")
print(
    X_test[0:5]
) 

Performing preprocessings on test set...
     model_key  mileage  engine_power    fuel paint_color   car_type  \
3203   Renault   109839           135  diesel       black      sedan   
1957  Mercedes   180032           105  diesel        grey  hatchback   
1044      Audi   147699           190  diesel        grey     estate   
2732   Renault    95241            85  diesel        blue      sedan   
1538       PGO   133214           190  diesel       black     estate   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
3203                          1        1                     0              0   
1957                          1        1                     0              0   
1044                          1        1                     0              1   
2732                          0        1                     0              0   
1538                          1        1                     0              0   

      has_getaround_connect  has_speed_

## 3. Linear Regression

In [26]:
# Train model
model = LinearRegression()
print("Training model...")
model.fit(X_train, y_train)  # Training is always done on train set !!
print("...Done.")

Training model...
...Done.


In [27]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 17.96
R²: 0.6937


## 4. Ridge Regression

In [29]:
ridge_model = Ridge()

In [32]:
param_grid_ridge = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}  #range of values to try for alpha

In [33]:
grid_search_ridge = GridSearchCV(ridge_model, param_grid_ridge, cv=5, scoring='neg_mean_squared_error')

In [34]:
grid_search_ridge.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.001, 0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [ ]:
print("Best param:", grid_search_ridge.best_params_)
print("Best score:", -grid_search_ridge.best_score_)  # minus sign: GridSearchCV use negatives scores for MSE


Best param: {'alpha': 1}
Best score: 342.9659451994612


In [36]:
best_ridge_model = grid_search_ridge.best_estimator_
best_ridge_model.fit(X_train, y_train)

Ridge(alpha=1)

In [46]:
y_pred = best_ridge_model.predict(X_test)

In [45]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 17.97
R²: 0.6935


## 5. Lasso Regression

In [47]:
lasso_model = Lasso()

In [48]:
param_grid_lasso = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]} 

In [49]:
grid_search_lasso = GridSearchCV(lasso_model, param_grid_lasso, cv=5, scoring='neg_mean_squared_error')
grid_search_lasso.fit(X_train, y_train)

c:\Users\Typhon\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:639: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 14079.76985073334, tolerance: 365.0160656340754
  model = cd_fast.sparse_enet_coordinate_descent(


GridSearchCV(cv=5, estimator=Lasso(),
             param_grid={'alpha': [0.001, 0.01, 0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [51]:
print("Best params:", grid_search_lasso.best_params_)
print("Best score:", -grid_search_lasso.best_score_)

Best params: {'alpha': 0.001}
Best score: 344.0543979629054


In [52]:
best_lasso_model = grid_search_lasso.best_estimator_
best_lasso_model.fit(X_train, y_train)

Lasso(alpha=0.001)

In [53]:
y_pred = best_lasso_model.predict(X_test)

In [54]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 17.97
R²: 0.6936


## 6. Random Forest

In [55]:
rf_model = RandomForestRegressor()

In [56]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [57]:
grid_search_rf = GridSearchCV(rf_model, param_grid=param_grid_rf, cv=5, scoring='neg_mean_squared_error')

In [58]:
grid_search_rf.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(),
             param_grid={'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='neg_mean_squared_error')

In [59]:
print("Best params:", grid_search_rf.best_params_)
print("Best score:", -grid_search_rf.best_score_)

Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Best score: 279.12860752556753


In [60]:
best_rf_model = grid_search_rf.best_estimator_
best_rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, min_samples_split=5, n_estimators=200)

In [61]:
y_pred = best_rf_model.predict(X_test)

In [62]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 16.76
R²: 0.7333


## 7. Gradient Boosting

In [63]:
gb_model = GradientBoostingRegressor(random_state=42)

In [64]:
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [66]:
grid_search_gb = GridSearchCV(gb_model, param_grid_gb, cv=5, scoring='neg_mean_squared_error')

In [67]:
grid_search_gb.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=GradientBoostingRegressor(random_state=42),
             param_grid={'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 4, 5], 'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='neg_mean_squared_error')

In [68]:
print("Best params:", grid_search_gb.best_params_)
print("Best score:", -grid_search_gb.best_score_)

Best params: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
Best score: 270.0418994928466


In [69]:
best_gb_model = grid_search_gb.best_estimator_
best_gb_model.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=5, min_samples_leaf=2, min_samples_split=10,
                          n_estimators=200, random_state=42)

In [70]:
y_pred = best_gb_model.predict(X_test)

In [71]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 16.01
R²: 0.7567


## 8. XGboost

In [22]:
xgb_model = xgb.XGBRegressor(random_state=42)

In [23]:
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

In [24]:
grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='neg_mean_squared_error') 

In [25]:
grid_search_xgb.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, gamma=None,
                                    grow_policy=None, importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None, m...
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None,
                                    random_state=42, ...),
             param_grid={'colsample_bytree': [0.7, 0.8, 0.9],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 4, 5], 'n_estimators': [50, 100, 200],
                         'subsample': [0.7, 0.8, 0.9]},
             scoring='neg_mean_squared_error')

In [26]:
print("Best params:", grid_search_xgb.best_params_)
print("Best score:", -grid_search_xgb.best_score_)

Best params: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.9}
Best score: 262.8620129649542


In [27]:
best_xgb_model = grid_search_xgb.best_estimator_
best_xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [28]:
y_pred = best_xgb_model.predict(X_test)

In [29]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

RMSE: 15.90
R²: 0.7600


In [30]:
import joblib

filename = 'xgb_model.joblib'
joblib.dump(best_xgb_model, filename)

['xgb_model.joblib']

In [ ]:
# Start MLflow experiment
with mlflow.start_run(run_name="XGBoost_Regression"):

    # Log preprocessor as artifact
    mlflow.log_artifact("preprocessor.joblib", artifact_path="preprocessor")

    # Model + Grid Search
    xgb_model = xgb.XGBRegressor(random_state=42)
    param_grid_xgb = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }
    grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='neg_mean_squared_error')
    grid_search_xgb.fit(X_train, y_train)

    best_xgb_model = grid_search_xgb.best_estimator_
    best_xgb_model.fit(X_train, y_train)

    # Evaluation
    y_pred = best_xgb_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("Best params:", grid_search_xgb.best_params_)
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")

    # Log parameters and metrics
    mlflow.log_params(grid_search_xgb.best_params_)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Log model
    mlflow.sklearn.log_model(best_xgb_model, artifact_path="model", registered_model_name="XGBRegressorModel")

